# 01 — Baselines and model selection

**Goal:** Start with simple statistical forecasts, evaluate them, and ask whether choosing a different baseline for each product improves the final result.

We use Nixtla **StatsForecast** and a 14-week holdout. All decisions are made on the earlier validation window—not on the final test window.

**Forecasts:** Naive, Historic Average, Seasonal Naive (52 weeks). The last model is evaluated only for series with at least 52 observed weeks at each origin.

## 1. Load the prepared dataset

Run this notebook from the repository root, or from `day1_ml/`. No price/promotion cleaning is needed yet: these baseline models only use historical sales.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import Naive, HistoricAverage, SeasonalNaive

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from shared.evaluation import scores, select_models, selected_forecasts

df = pd.read_parquet(ROOT / "data/retail_prepared.parquet")
df = df[["unique_id", "ds", "y"]].copy()
df["ds"] = pd.to_datetime(df["ds"])
df = df.sort_values(["unique_id", "ds"])
print(f"{len(df):,} rows | {df.unique_id.nunique():,} series")

## 2. Validation and test windows

Our final 14 weeks (April 16–July 16, 2025) are the **test set**. The preceding 14 weeks (January 8–April 9, 2025) are the **validation set**. A model is selected using validation only.

We generate each forecast from data available at its forecast origin. A baseline cannot forecast a product with *no training history*; we report those omissions rather than using future data to construct predictions.

In [ ]:
H = 14
VALID_START = pd.Timestamp("2025-01-08")
TEST_START = pd.Timestamp("2025-04-16")
MODELS = ["Naive", "HistoricAverage", "SeasonalNaive"]

valid_train = df[df.ds < VALID_START]
valid_actual = df[(df.ds >= VALID_START) & (df.ds < TEST_START)]
test_train = df[df.ds < TEST_START]
test_actual = df[df.ds >= TEST_START]

print(f"Validation: {valid_actual.ds.min():%Y-%m-%d} to {valid_actual.ds.max():%Y-%m-%d}")
print(f"Test:       {test_actual.ds.min():%Y-%m-%d} to {test_actual.ds.max():%Y-%m-%d}")

## 3. Forecast the baseline models

Nixtla expects `unique_id`, `ds`, and `y`. We use `forecast()` because these models don't need a separate fitted object for this exercise.

Series with fewer than 52 historical weeks get Naive and Historic Average, but **not** a Seasonal Naive score. This avoids disguising a different model as a seasonal forecast.

In [ ]:
def baseline_forecasts(train, actual):
    # Forecast only series with a complete 14-week evaluation window.
    counts = actual.groupby("unique_id").size()
    eligible = counts[counts == H].index
    history = train[train.unique_id.isin(eligible)]

    sf = StatsForecast(models=[Naive(), HistoricAverage()], freq="W-WED", n_jobs=-1)
    basic = sf.forecast(df=history, h=H)

    lengths = history.groupby("unique_id").size()
    seasonal_ids = lengths[lengths >= 52].index
    seasonal = StatsForecast(
        models=[SeasonalNaive(season_length=52)], freq="W-WED", n_jobs=-1
    ).forecast(df=history[history.unique_id.isin(seasonal_ids)], h=H)

    return (actual[actual.unique_id.isin(eligible)]
            .merge(basic, on=["unique_id", "ds"], how="inner")
            .merge(seasonal, on=["unique_id", "ds"], how="left"))

valid = baseline_forecasts(valid_train, valid_actual)
test = baseline_forecasts(test_train, test_actual)
print(f"Validation: {valid.unique_id.nunique():,} series")
print(f"Test: {test.unique_id.nunique():,} series")

## 4. Compare standalone baselines

**WAPE** = sum of absolute errors / sum of actual demand. **Accuracy** = 1 − WAPE. **Bias** = sum(forecast − actual) / sum(actual). These are volume-weighted portfolio statistics.

Because Seasonal Naive is unavailable for some series, compare models on a **common eligible set** before interpreting rankings.

In [ ]:
common_ids = set(valid.unique_id) & set(test.unique_id)
seasonal_ids = set(valid.loc[valid.SeasonalNaive.notna(), "unique_id"])
common_ids &= seasonal_ids

valid_common = valid[valid.unique_id.isin(common_ids)].copy()
test_common = test[test.unique_id.isin(common_ids)].copy()

print(f"Common series with 52+ weeks at validation: {len(common_ids):,}")
display(scores(valid_common, MODELS).round(4))

## 5. Choose a baseline using validation only

For every product, select the baseline with the **lowest validation MAE**. This is a simple selection rule—not an optimized or statistically corrected tournament.

Compare it with individual baselines on the **same series**, then apply the stored selections to the untouched test window.

In [ ]:
chosen = select_models(valid_common, MODELS)
print(chosen.value_counts().rename("Products selected"))

valid_selected = selected_forecasts(valid_common, chosen, MODELS)
test_selected = selected_forecasts(test_common, chosen, MODELS)

print("Validation")
display(scores(valid_selected, MODELS + ["Selected"]).round(4))
print("Test")
display(scores(test_selected, MODELS + ["Selected"]).round(4))

## 6. Visualize the final comparison

Does selection still win on the unseen test window—or did it exploit noise in validation?

In [ ]:
leaderboard = scores(test_selected, MODELS + ["Selected"])
ax = leaderboard.set_index("model")["WAPE"].plot.bar(figsize=(8, 4), rot=0)
ax.set(title="Test WAPE: individual baselines vs. validation selection", ylabel="WAPE", xlabel="")
plt.tight_layout()
plt.show()

## 7. Save results for the ML and neural-model workshops

These files allow later labs to reuse the same forecasts and selection outcomes without refitting baselines.

In [ ]:
results = ROOT / "results"
results.mkdir(exist_ok=True)

valid_selected.assign(split="validation").to_parquet(results / "baseline_validation.parquet", index=False)
test_selected.assign(split="test").to_parquet(results / "baseline_test.parquet", index=False)
chosen.to_csv(results / "baseline_selection.csv")
leaderboard.to_csv(results / "baseline_leaderboard.csv", index=False)

print(f"Saved baseline results to {results}")

### Discuss

- Why might the validation-selected forecasts look better on validation but fail to improve the test result?
- Why isn't the baseline winner the same for every product?
- What changes when we add LightGBM to the selection pool?

**Next:** Exogenous drivers and MLForecast, then repeat the same selection experiment with ML candidates.